# Test5: Qwen3-14B with the DGIdb-grounded v3 prompt

Replicate test4's complete curated 100-interaction workflow, using the current inclusive
v3 prompt, packaged DGIdb references and canonical result schema. Run this notebook
from this directory. Install from the repository root with `pip install -e '.[qwen,pmc]'`;
the notebook also needs pandas, python-dotenv, openpyxl, fastexcel, ipywidgets and xlsxwriter.
Use the same `PUBTATOR_PATH` as test2/test4. Inputs and earlier test outputs are read-only;
all test5 outputs are written here. Model inference uses MLX/Metal on Apple Silicon.

Run `01` before `02_analysis.ipynb` and `03_concordance.ipynb`. Classification failures
are exported with `interaction=null`, never silently relabeled as negative. The separate
`04_reviewed_seed_analysis.ipynb` displays the already-completed 42-case human-reviewed
seed experiment; it is not the full curated 100-interaction experiment.


In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import importlib.metadata

# Prefer this checkout even if another dgilit checkout is installed editable.
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "src/dgilit/dgilit_models.py").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))
OUTPUT_DIR = REPO_ROOT / "notebooks/test_sets/test5_dgidb_qwen_v3_prompt"

from itertools import product
from dotenv import load_dotenv
import os


# Avoid the Xet download route that appears to have stalled.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

load_dotenv()

PUBTATOR_PATH = os.environ["PUBTATOR_PATH"]


import pandas as pd
import polars as pl
from tqdm.notebook import tqdm

# wags-llm imports
from wags_llm.cache import InMemoryCache
from dgilit import Qwen3JsonClient
from wags_llm.prompts import build_empty_registry
from wags_llm.services import StructuredTaskRunner

# dgiLIT imports
from dgilit import (
    BioBertEntityTagger,
    CompositeEntityTagger,
    EntityPreTaggingService,
    InteractionClassificationPrompt,
    InteractionClassificationResult,
    classification_result_model,
    validate_classification,
    PubTator3ChemicalTagger,
    SQLiteNormalizerCache,
    TaggerConfig,
    ViccNormalizer,
)

## Qwen model setup

In [ ]:
MODEL_ID = "mlx-community/Qwen3-14B-4bit"
MAX_TOKENS = 1000
PROMPT_VERSION = "v3"
prompt = InteractionClassificationPrompt(PROMPT_VERSION)
SYSTEM_PROMPT = prompt.build_system_prompt()  # Fail before model load if references are missing.
from dgilit.pmc_store import digest
PROMPT_HASH = digest(SYSTEM_PROMPT)
from dgilit.pmc_pipeline import _model_revision

(OUTPUT_DIR / "v3_system_prompt.text").write_text(SYSTEM_PROMPT)
metadata = {
    "model_id": MODEL_ID, "model_revision": _model_revision(MODEL_ID),
    "prompt_version": PROMPT_VERSION, "prompt_hash": PROMPT_HASH,
    "max_tokens": MAX_TOKENS, "temperature": 0, "enable_thinking": False,
    "input_workbook_sha256": hashlib.sha256((REPO_ROOT / "data/2026-06-29-test2-RohitList-wText.xlsx").read_bytes()).hexdigest(),
    "packages": {name: importlib.metadata.version(name) for name in ("mlx", "mlx-lm", "pydantic", "wags-llm")},
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n")

# Load once; task runners below share weights but have independent caches.
llm_client = Qwen3JsonClient(
    model_id=MODEL_ID,
    max_tokens=MAX_TOKENS, temperature=0.0,
)

def build_llm_task_runner(temperature, prompt_version):
    llm_client.temperature = temperature
    registry = build_empty_registry()
    registry.register(InteractionClassificationPrompt(version=prompt_version))
    return StructuredTaskRunner(
        client=llm_client, prompt_registry=registry, cache=InMemoryCache()
    )

print(f"Model load: {llm_client.load_seconds:.2f}s")
print(f"Backend: {llm_client.backend}, device: {llm_client.device}, weights: {llm_client.quantization_bits}-bit")


In [ ]:
# Fail before the full run if model generation or schema validation fails.
smoke_prompt = InteractionClassificationPrompt(version="v3")
smoke_result = build_llm_task_runner(0.0, "v3").execute(
    prompt_name=smoke_prompt.name,
    prompt_version=smoke_prompt.version,
    payload=smoke_prompt.build_payload(
        context="Venetoclax binds BCL2 and inhibits its activity.",
        candidate_drug="Venetoclax", candidate_gene="BCL2",
    ),
    response_model=classification_result_model(smoke_prompt.version),
)
smoke_result = validate_classification(smoke_result, version=smoke_prompt.version,
    context="Venetoclax binds BCL2 and inhibits its activity.", candidate_drug="Venetoclax", candidate_gene="BCL2")
assert smoke_result.interaction, "Smoke test did not recover the explicit target interaction"
print(llm_client.last_generation_stats)
smoke_result


## Input and pre-tagging

In [ ]:
def classify_interaction(task_runner, prompt, context, candidate_drug, candidate_gene):
    """Return a validated decision or a separate failure record; never a fabricated negative."""
    payload = prompt.build_payload(context, candidate_drug, candidate_gene)
    try:
        result = task_runner.execute(
            prompt_name=prompt.name, prompt_version=prompt.version, payload=payload,
            response_model=classification_result_model(prompt.version),
        )
        result = validate_classification(result, version=prompt.version, context=context,
                                        candidate_drug=candidate_drug, candidate_gene=candidate_gene)
        return {**result.model_dump(), "classification_status": "valid"}
    except Exception as exc:
        return {"drug": candidate_drug, "gene": candidate_gene, "interaction": None,
                "evidence": None, "interaction_type": None, "directionality": None,
                "error_message": str(exc), "classification_status": "failed"}


In [ ]:
df = pl.read_excel("../../../data/2026-06-29-test2-RohitList-wText.xlsx") # file with pmids + context blocks

In [ ]:
tagger = CompositeEntityTagger(
    BioBertEntityTagger(
        TaggerConfig(
            batch_size=16,
            include_drugs=True,
            include_genes=True,
            include_diseases=False,
            device=-1,
        )
    ),
    PubTator3ChemicalTagger.from_pubtator_file(PUBTATOR_PATH),
)

pretagger = EntityPreTaggingService(
    tagger=tagger,
    normalizer=ViccNormalizer(
        cache=SQLiteNormalizerCache(".dgilit_normalizer_cache.sqlite")
    ),
)

In [ ]:
contexts = (
    df["context"]
    .fill_null("")
    .cast(pl.Utf8)
    .to_list()
)
pmids = df["pmid"].cast(str).to_list() if "pmid" in df.columns else [None] * len(contexts)
block_ids = [str(i) for i in range(len(contexts))]

tagged_blocks = pretagger.tag_blocks(
    contexts=contexts,
    pmids=pmids,
    block_ids=block_ids,
)

In [ ]:
def entity_display_name(e):
    if e.concept and e.concept.concept_label:
        return e.concept.concept_label

    if e.text:
        return e.text

    return None


def build_candidate_context(block):
    drugs = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "drug"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    genes = sorted({
        e.concept.concept_label
        for e in block.entities
        if e.entity_type == "gene"
        and e.concept
        and e.concept.concept_label
        and e.concept.concept_id
    })

    return {
        "pmid": block.pmid,
        "drugs": drugs,
        "genes": genes,
        "context": block.context,
    }

candidate = build_candidate_context(tagged_blocks[0])
candidate

## LLM Calls

Description

In [ ]:
# EXPERIMENTAL
def normalized_entity_name(e):
    if e.concept and e.concept.concept_label and e.concept.concept_id:
        return e.concept.concept_label
    return None


def raw_entity_name(e):
    if e.text:
        return e.text
    return None


def get_candidates(block):
    """Normalized candidates only."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [normalized_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [normalized_entity_name(e)]
        if name
    })

    return drugs, genes


def get_raw_candidates(block):
    """Raw mentions regardless of normalization."""
    drugs = sorted({
        name
        for e in block.entities
        if e.entity_type == "drug"
        for name in [raw_entity_name(e)]
        if name
    })

    genes = sorted({
        name
        for e in block.entities
        if e.entity_type == "gene"
        for name in [raw_entity_name(e)]
        if name
    })

    return drugs, genes

# Inspect one tagged block before inference.
get_candidates(tagged_blocks[0])


In [ ]:
import polars as pl
from itertools import product
from tqdm import tqdm


def make_pmid_to_genes(df: pl.DataFrame) -> dict:
    pmid_to_genes_df = (
        df
        .with_columns(pl.col("pmid").cast(pl.Utf8))
        .group_by("pmid")
        .agg(pl.col("gene_name").drop_nulls().unique().sort())
    )

    return dict(
        zip(
            pmid_to_genes_df["pmid"].to_list(),
            pmid_to_genes_df["gene_name"].to_list(),
        )
    )


def run_experiments(
    tagged_blocks,
    temperatures,
    num_runs,
    prompt_version: str,
    use_raw_drug_candidates: bool = False,
    seed_gene: str | None = None,
    pmid_to_genes: dict | None = None,
):
    stored_runs = []

    for temp in temperatures:
        for run_idx in range(num_runs):
            task_runner = build_llm_task_runner(
                temperature=temp, prompt_version=prompt_version,
            )

            prompt = InteractionClassificationPrompt(version=prompt_version)

            print(f"Running temp={temp}, run={run_idx + 1}")

            results = []

            for block in tqdm(
                tagged_blocks,
                total=len(tagged_blocks),
                desc=f"T={temp}, run={run_idx + 1}",
                leave=False,
            ):
                normalized_drugs, normalized_genes = get_candidates(block)
                raw_drugs, raw_genes = get_raw_candidates(block)

                if use_raw_drug_candidates:
                    candidate_drugs = sorted(set(normalized_drugs) | set(raw_drugs))
                else:
                    candidate_drugs = normalized_drugs

                if pmid_to_genes is not None:
                    candidate_genes_for_classification = pmid_to_genes.get(
                        str(block.pmid),
                        [],
                    )
                elif seed_gene:
                    candidate_genes_for_classification = [seed_gene]
                else:
                    candidate_genes_for_classification = normalized_genes

                if not candidate_drugs or not candidate_genes_for_classification:
                    results.append(
                        {
                            "pmid": block.pmid,
                            "block_id": block.block_id,
                            "skipped": True,
                            "skip_reason": "No drug and gene candidates",
                            "candidate_drugs": candidate_drugs,
                            "candidate_genes": candidate_genes_for_classification,
                            "normalized_drugs": normalized_drugs,
                            "normalized_genes": normalized_genes,
                            "raw_drugs": raw_drugs,
                            "raw_genes": raw_genes,
                            "classifications": [],
                        }
                    )
                    continue

                classifications = []

                for candidate_drug, candidate_gene in product(
                    candidate_drugs,
                    candidate_genes_for_classification,
                ):
                    result = classify_interaction(
                        task_runner=task_runner,
                        prompt=prompt,
                        context=block.context,
                        candidate_drug=candidate_drug,
                        candidate_gene=candidate_gene,
                    )

                    classifications.append(result)

                results.append(
                    {
                        "pmid": block.pmid,
                        "block_id": block.block_id,
                        "skipped": False,
                        "candidate_drugs": candidate_drugs,
                        "candidate_genes": candidate_genes_for_classification,
                        "normalized_drugs": normalized_drugs,
                        "normalized_genes": normalized_genes,
                        "raw_drugs": raw_drugs,
                        "raw_genes": raw_genes,
                        "num_candidate_pairs": len(candidate_drugs)
                        * len(candidate_genes_for_classification),
                        "classifications": classifications,
                    }
                )

            stored_runs.append(
                {
                    "model_id": MODEL_ID,
                    "max_tokens": MAX_TOKENS,
                    "enable_thinking": False,
                    "backend": llm_client.backend,
                    "device": llm_client.device,
                    "quantization_bits": llm_client.quantization_bits,
                    "run_idx": run_idx,
                    "prompt_version": prompt_version,
                    "prompt_hash": PROMPT_HASH,
                    "temperature": temp,
                    "use_raw_drug_candidates": use_raw_drug_candidates,
                    "seed_gene": seed_gene,
                    "used_pmid_to_genes": pmid_to_genes is not None,
                    "results": results,
                }
            )

            print(f"Done temp={temp}, run={run_idx + 1}")

    return stored_runs

In [ ]:
# Representative check using the same candidate selection as the full experiment.
benchmark_runner = build_llm_task_runner(0.0, PROMPT_VERSION)
benchmark_prompt = InteractionClassificationPrompt(PROMPT_VERSION)
benchmark_genes = make_pmid_to_genes(df)
for block in tagged_blocks:
    drugs, _ = get_candidates(block)
    genes = benchmark_genes.get(str(block.pmid), [])
    if drugs and genes:
        benchmark_result = classify_interaction(benchmark_runner, benchmark_prompt,
                                               block.context, drugs[0], genes[0])
        print(f"PMID={block.pmid}, drug={drugs[0]}, gene={genes[0]}")
        print(llm_client.last_generation_stats)
        print(benchmark_result)
        break
else:
    raise RuntimeError("No representative candidate pair available")


In [ ]:
pmid_to_genes = make_pmid_to_genes(df)

stored_runs = run_experiments(
    tagged_blocks=tagged_blocks,
    temperatures=[0],
    num_runs=1,
    prompt_version="v3",
    use_raw_drug_candidates=False,
    seed_gene=None,
    pmid_to_genes=pmid_to_genes,
)

## Output

In [ ]:
seen = set()

for run in stored_runs:
    for result in run["results"]:
        key = (result["pmid"], tuple(result["candidate_genes"]))

        if key in seen:
            continue

        seen.add(key)

        print(
            f"PMID={result['pmid']} | "
            f"candidate_genes={result['candidate_genes']}"
        )

In [ ]:
def save_experiment_results(
    stored_runs,
    csv_path="interaction_classifications-rohit-qwen3-14b-v3.csv",
    parquet_path="interaction_classifications-rohit-qwen3-14b-v3.parquet",
):
    rows = []

    for run in stored_runs:
        for result in run["results"]:
            candidate_drugs = result.get("candidate_drugs", [])
            candidate_genes = result.get("candidate_genes", [])
            classifications = result.get("classifications", [])

            normalized_drugs = result.get("normalized_drugs", [])
            normalized_genes = result.get("normalized_genes", [])
            raw_drugs = result.get("raw_drugs", [])
            raw_genes = result.get("raw_genes", [])

            used_pmid_to_genes = run.get("used_pmid_to_genes", False)

            base_row = {
                "model_id": run.get("model_id"),
                "max_tokens": run.get("max_tokens"),
                "enable_thinking": run.get("enable_thinking"),
                "backend": run.get("backend"),
                "device": run.get("device"),
                "quantization_bits": run.get("quantization_bits"),
                "run_idx": run.get("run_idx"),
                "prompt_version": run.get("prompt_version"),
                "prompt_hash": run.get("prompt_hash"),
                "temperature": run.get("temperature"),
                "used_raw_drug_candidates": run.get(
                    "use_raw_drug_candidates",
                    False,
                ),
                "seed_gene": run.get("seed_gene"),
                "used_pmid_to_genes": used_pmid_to_genes,
                "gene_source": "pmid_to_genes" if used_pmid_to_genes else (
                    "seed_gene" if run.get("seed_gene") else "normalized_genes"
                ),

                "pmid": result.get("pmid"),
                "block_id": result.get("block_id"),
                "skipped": result.get("skipped", False),
                "skip_reason": result.get("skip_reason"),

                # These are the actual candidates used for classification.
                "candidate_drugs": ";".join(candidate_drugs),
                "candidate_genes": ";".join(candidate_genes),
                "candidate_drug_count": len(candidate_drugs),
                "candidate_gene_count": len(candidate_genes),
                "num_candidate_pairs": result.get("num_candidate_pairs", 0),

                # These are tagger-derived candidates, kept for debugging.
                "normalized_drugs": ";".join(normalized_drugs),
                "normalized_genes": ";".join(normalized_genes),
                "normalized_drug_count": len(normalized_drugs),
                "normalized_gene_count": len(normalized_genes),

                "raw_drugs": ";".join(raw_drugs),
                "raw_genes": ";".join(raw_genes),
                "raw_drug_count": len(raw_drugs),
                "raw_gene_count": len(raw_genes),

                "result_error_message": result.get("error_message"),
            }

            if not classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": None,
                        "gene": None,
                        "interaction": None,
                        "interaction_type": None,
                        "directionality": None,
                        "evidence": None,
                        "classification_error_message": None,
                        "classification_status": "skipped",
                        "error_message": result.get("error_message"),
                    }
                )
                continue

            for classification in classifications:
                rows.append(
                    {
                        **base_row,
                        "drug": classification.get("drug"),
                        "gene": classification.get("gene"),
                        "interaction": classification.get("interaction"),
                        "interaction_type": classification.get("interaction_type"),
                        "directionality": classification.get("directionality"),
                        "evidence": classification.get("evidence"),
                        "classification_status": classification.get("classification_status"),
                        "classification_error_message": classification.get(
                            "error_message"
                        ),
                        "error_message": classification.get("error_message")
                        or result.get("error_message"),
                    }
                )

    df_results = pl.DataFrame(rows)

    df_results.write_csv(csv_path)
    df_results.write_parquet(parquet_path)

    print(df_results.shape)
    return df_results


df_results = save_experiment_results(stored_runs)

df_results.head()